# Day 2 — 포트폴리오 분석

Day 1에서 수집한 데이터를 바탕으로  
가상의 포트폴리오를 구성하고, 90일간의 수익을 추적합니다.

---

## 포트폴리오란?

여러 종목에 분산해서 투자한 자산 묶음입니다.  
한 종목만 사면 그 종목이 폭락할 때 전부 잃지만,  
여러 종목에 나눠 투자하면 리스크를 줄일 수 있어요.

```
초기 자본금: 10,000,000 원
├── KRW-BTC  40%  →  4,000,000 원
├── KRW-ETH  30%  →  3,000,000 원
├── KRW-SOL  20%  →  2,000,000 원
└── KRW-XRP  10%  →  1,000,000 원
```

## MDD(Maximum Drawdown)란?

투자 기간 중 고점 대비 **가장 많이 떨어진 낙폭**입니다.  
예) MDD -20% → 한때 최고 자산에서 20% 빠진 적 있었다는 의미  
리스크를 측정하는 핵심 지표입니다.

In [ ]:
%run 01_data_pipeline.ipynb
# Day 1 실행: data 딕셔너리(4종목 180일치)와 함수들을 모두 가져옴

## 1. 포트폴리오 설정

In [ ]:
# ── 포트폴리오 설정 ─────────────────────────────────────────────
# 딕셔너리 안에 딕셔너리를 넣어서 종목별 정보를 구조화
PORTFOLIO = {
    "KRW-BTC": {"weight": 0.4, "amount": 4_000_000},  # 4백만 원
    "KRW-ETH": {"weight": 0.3, "amount": 3_000_000},  # 3백만 원
    "KRW-SOL": {"weight": 0.2, "amount": 2_000_000},  # 2백만 원
    "KRW-XRP": {"weight": 0.1, "amount": 1_000_000},  # 1백만 원
}

INITIAL_CAPITAL = 10_000_000  # 초기 자본금 1천만 원
FEE_RATE        = 0.0005      # 수수료 0.05% (매수 시 1회)
START_DAYS_AGO  = 90          # 투자 시작: 90일 전

print(f"초기 자본금 : {INITIAL_CAPITAL:,} 원")
print(f"투자 수수료 : {FEE_RATE*100:.2f}%")
print(f"투자 기간   : {START_DAYS_AGO}일")
print()
for ticker, cfg in PORTFOLIO.items():
    print(f"  {ticker}  {cfg['weight']*100:.0f}%  →  {cfg['amount']:,} 원")

## 2. 일별 자산 추적

### 흐름 요약
1. 90일 전 종가로 각 종목의 **보유 수량** 계산 (수수료 차감 후)
2. 이후 매일: `총 자산 = Σ(보유 수량 × 해당일 종가)`
3. 리밸런싱(비중 재조정) 없음 → 수량 고정

In [ ]:
def get_portfolio_data(data, start_days_ago=90):
    """
    전체 데이터(180일)에서 포트폴리오 분석 기간(90일)만 잘라서 반환합니다.
    tail(90): 마지막 90개 행 = 최근 90일
    """
    port_data = {}
    for ticker in data:
        port_data[ticker] = data[ticker].tail(start_days_ago).copy()
    return port_data

In [ ]:
def calc_holdings(port_data, portfolio, fee_rate):
    """
    투자 시작일(90일 전) 종가 기준으로 각 종목 보유 수량을 계산합니다.
    보유 수량 = 투자금액 × (1 - 수수료율) ÷ 시작일 종가

    반환값:
        {"KRW-BTC": 0.01703, "KRW-ETH": 1.234, ...}  ← 코인 수량
    """
    holdings = {}
    for ticker, config in portfolio.items():
        if ticker not in port_data:
            continue

        # iloc[0] : 첫 번째 행 = 90일 전 (포트폴리오 시작일) 종가
        start_price = port_data[ticker].iloc[0]["close"]

        # 수수료를 차감한 실제 투자금
        invested = config["amount"] * (1 - fee_rate)

        # 보유 수량 = 투자금 ÷ 단가
        holdings[ticker] = invested / start_price

    return holdings

In [ ]:
def calc_daily_values(port_data, holdings):
    """
    날짜별 포트폴리오 총 자산을 계산합니다.
    총 자산 = Σ(보유 수량 × 해당일 종가)

    반환값:
        pandas Series (날짜가 인덱스, 총 자산이 값)
    """
    # BTC 날짜를 기준으로 사용 (4종목 날짜가 동일)
    ref = list(holdings.keys())[0]
    dates = port_data[ref].index

    daily_totals = []
    for date in dates:
        total = 0
        for ticker, qty in holdings.items():
            # loc[date, 'close'] : 특정 날짜의 종가 가져오기
            price = port_data[ticker].loc[date, "close"]
            total += qty * price
        daily_totals.append(total)

    return pd.Series(daily_totals, index=dates, name="portfolio_value")

In [ ]:
def calc_mdd(series):
    """
    MDD(Maximum Drawdown, 최대 낙폭)를 계산합니다.
    MDD = 고점 대비 가장 크게 빠진 비율 (%)

    예) MDD -20% → 투자 기간 중 한때 고점에서 20% 하락한 적 있음
    """
    # cummax() : 해당 시점까지의 최대값 (고점)
    rolling_max = series.cummax()

    # (현재값 - 고점) / 고점 = 고점 대비 하락률
    drawdown = (series - rolling_max) / rolling_max * 100

    # .min() : 가장 많이 빠진 시점의 값
    return drawdown.min()

In [ ]:
def calc_ticker_contribution(port_data, portfolio, holdings):
    """
    종목별 수익률과 포트폴리오 기여도를 계산합니다.

    기여도 = 종목 수익률 × 포트폴리오 내 비중
    예) BTC가 +10% 올랐고 비중이 40%면 → 기여도 +4%p
    """
    result = {}
    for ticker, config in portfolio.items():
        if ticker not in port_data:
            continue
        start_price = port_data[ticker].iloc[0]["close"]
        end_price   = port_data[ticker].iloc[-1]["close"]

        # 종목 수익률 (%)
        ticker_return = (end_price - start_price) / start_price * 100

        # 기여도 = 수익률 × 비중 (%p)
        contribution = ticker_return * config["weight"]

        result[ticker] = {
            "return":       ticker_return,
            "contribution": contribution,
            "weight":       config["weight"]
        }
    return result

In [ ]:
def print_portfolio_summary(portfolio_values, portfolio, port_data):
    """
    포트폴리오 전체 성과 요약을 출력합니다.
    """
    start_val = portfolio_values.iloc[0]
    end_val   = portfolio_values.iloc[-1]
    total_ret = (end_val - start_val) / start_val * 100
    mdd       = calc_mdd(portfolio_values)

    start_date = portfolio_values.index[0].strftime("%Y-%m-%d")
    end_date   = portfolio_values.index[-1].strftime("%Y-%m-%d")

    holdings = calc_holdings(port_data, portfolio, FEE_RATE)
    contrib  = calc_ticker_contribution(port_data, portfolio, holdings)

    print("=== 포트폴리오 성과 요약 ===")
    print(f"투자 기간    : {start_date} ~ {end_date} ({START_DAYS_AGO}일)")
    print(f"초기 자산    : {INITIAL_CAPITAL:,} 원")
    print(f"현재 자산    : {end_val:,.0f} 원")
    print(f"총 수익률    : {total_ret:+.2f}%")
    print(f"MDD         : {mdd:.2f}%")
    print()
    print("종목별 기여도:")
    for ticker, info in contrib.items():
        print(f"  {ticker}  수익률 {info['return']:+.2f}%  기여 {info['contribution']:+.2f}%p")

In [ ]:
# ── 실행 ─────────────────────────────────────────────────────────
port_data        = get_portfolio_data(data, START_DAYS_AGO)
holdings         = calc_holdings(port_data, PORTFOLIO, FEE_RATE)
portfolio_values = calc_daily_values(port_data, holdings)

print_portfolio_summary(portfolio_values, PORTFOLIO, port_data)

## 3. 시각화

### subplot 3개 구성
- **상단**: 일별 총 자산 곡선
- **중단**: 종목별 누적 수익률 비교 (시작일 = 100으로 정규화)
- **하단**: 일별 수익률 바차트 (양수 빨강 / 음수 파랑)

### 정규화란?
서로 단위가 다른 종목(BTC 1억, XRP 2천원)을 같은 기준으로 비교하는 방법입니다.  
시작일 가격을 모두 100으로 맞추면 등락률을 같은 눈금으로 비교할 수 있어요.

In [ ]:
def make_asset_curve_trace(portfolio_values):
    """상단: 일별 총 자산 곡선 trace 반환"""
    return go.Scatter(
        x=portfolio_values.index,
        y=portfolio_values.values,
        line=dict(color="navy", width=2),
        fill="tozeroy",               # y=0까지 채우기 (면적 차트)
        fillcolor="rgba(0,0,128,0.1)",
        name="총 자산"
    )

In [ ]:
def make_normalized_traces(port_data):
    """
    중단: 종목별 누적 수익률 비교 traces 반환.
    시작일 종가를 100으로 맞춰서 등락률을 같은 기준으로 비교합니다.
    """
    colors = {"KRW-BTC": "orange", "KRW-ETH": "blue",
              "KRW-SOL": "green",  "KRW-XRP": "red"}
    traces = []
    for ticker, df in port_data.items():
        start_price = df["close"].iloc[0]
        # 시작가 = 100 기준으로 정규화
        normalized = df["close"] / start_price * 100
        traces.append(go.Scatter(
            x=df.index, y=normalized,
            line=dict(color=colors.get(ticker, "gray"), width=1.5),
            name=ticker
        ))
    return traces

In [ ]:
def make_daily_return_traces(portfolio_values):
    """
    하단: 일별 수익률 바차트 traces 반환.
    양수(오름) → 빨강, 음수(내림) → 파랑
    """
    daily_ret = portfolio_values.pct_change() * 100  # 일별 수익률 (%)
    colors    = ["red" if r >= 0 else "blue" for r in daily_ret]

    return [go.Bar(
        x=daily_ret.index,
        y=daily_ret.values,
        marker_color=colors,
        name="일별 수익률"
    )]

In [ ]:
def make_portfolio_chart(portfolio_values, port_data):
    """
    포트폴리오 시각화: 자산곡선 / 누적수익률 / 일별수익률 3개 subplot
    """
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=["총 자산 곡선", "종목별 누적 수익률 (시작=100)", "일별 수익률"],
        vertical_spacing=0.10,
        row_heights=[0.4, 0.35, 0.25]  # 각 subplot 높이 비율
    )

    # 상단: 자산 곡선
    fig.add_trace(make_asset_curve_trace(portfolio_values), row=1, col=1)

    # 중단: 종목별 정규화 수익률
    for trace in make_normalized_traces(port_data):
        fig.add_trace(trace, row=2, col=1)

    # 하단: 일별 수익률 바차트
    for trace in make_daily_return_traces(portfolio_values):
        fig.add_trace(trace, row=3, col=1)

    fig.update_layout(
        title="포트폴리오 분석 (최근 90일)",
        height=900,
        template="plotly_white"
    )
    # y축 포맷: 상단은 원화, 중단은 지수, 하단은 %
    fig.update_yaxes(tickformat=",", row=1, col=1)
    fig.update_yaxes(ticksuffix="%", row=3, col=1)
    return fig

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = "AppleGothic"  # 한글 폰트 (macOS)
matplotlib.rcParams["axes.unicode_minus"] = False    # 마이너스 부호 깨짐 방지

def make_correlation_heatmap(port_data):
    """
    4종목 일별 수익률의 상관관계 히트맵을 출력합니다.

    상관계수란?
        1에 가까울수록 → 같은 방향으로 움직임
        0에 가까울수록 → 서로 독립적
       -1에 가까울수록 → 반대 방향으로 움직임
    분산 투자 효과는 상관계수가 낮을수록 커집니다.
    """
    # 종목별 일별 수익률을 하나의 DataFrame으로 합치기
    returns_df = pd.DataFrame({
        ticker: port_data[ticker]["close"].pct_change() * 100
        for ticker in port_data
    }).dropna()  # NaN 행 제거

    # 상관계수 행렬 계산
    corr = returns_df.corr()

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        corr,
        annot=True,        # 셀 안에 숫자 표시
        fmt=".2f",         # 소수점 2자리
        cmap="RdYlGn",     # 빨강(음)-노랑(0)-초록(양) 색상
        vmin=-1, vmax=1,   # 색상 범위 -1 ~ 1
        ax=ax
    )
    ax.set_title("4종목 일별 수익률 상관관계")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 차트 출력 ─────────────────────────────────────────────────────
fig = make_portfolio_chart(portfolio_values, port_data)
fig.show()

make_correlation_heatmap(port_data)